# 03 Environmental Constraints

Combine saved **2026 summer NDVI**, **2024 NLCD land cover** (or the recorded analysis year), elevation, slope, and FEMA SFHA coverage for the seven confirmed parcels. This notebook runs locally from completed source outputs; no Earth Engine connection is needed.

Run `01_Vegetation`, `01b_Land_Cover_Development`, `02_Terrain`, and `02b_Flooding` first. Outputs are saved in `goat_island_outputs/environmental_constraints`. Figures are PNG only.

The result is an **exploratory disturbance and physical-constraint screening score**, not a validated habitat-quality measure, a legal development restriction, or final recreational access suitability. Land-cover scores and weights below are explicit, editable assumptions. Different interventions may need different rules.

NLCD maps broad 30 m land-cover classes; its forest/wetland categories do not independently establish habitat quality or jurisdictional wetland status. Consult the [USGS Annual NLCD documentation](https://www.usgs.gov/centers/eros/science/nlcd-product-suite) for source definitions. Vegetation greenness and land-cover classes overlap conceptually, as do low elevation and flood exposure; sensitivity scenarios help inspect these assumptions. FEMA remains a binary SFHA factor in this model; floodway detail is available in the flooding notebook.

## 1. Imports

Use local rasters, vectors, and recorded metadata.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
from rasterio.plot import show
from rasterio.warp import reproject, Resampling
from shapely import force_2d
from IPython.display import display


## 2. Paths and modeling assumptions

Higher land-cover scores represent greater provisional caution about disturbance. Lower developed-land scores represent existing surface modification; other site hazards remain separate factors. Scores are assigned to classes, not learned from field observations.

In [ ]:
PROJECT_ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "Park_Parcels/Park_Parcles.shp").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside goat-island-analysis.")
OUTPUT_DIR = PROJECT_ROOT / "goat_island_outputs"
MODEL_DIR = OUTPUT_DIR / "environmental_constraints"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_CRS = "EPSG:26914"
NODATA = -9999
PARCEL_IDS = sorted(
    [
        "65079661510030000",
        "65130905010020000",
        "65130905010030000",
        "65130905010050000",
        "65130905010060000",
        "65146334010100000",
        "65146334010140000",
    ]
)

# Provisional class scores for a disturbance-screening scenario, not ecological measurements.
LAND_COVER_RULES = {
    11: ("Open water", 1.00, "High caution for terrestrial access interventions"),
    12: (
        "Perennial ice/snow",
        1.00,
        "High caution; unexpected locally and requires review",
    ),
    21: (
        "Developed, open space",
        0.30,
        "Already modified, potentially vegetated surface",
    ),
    22: (
        "Developed, low intensity",
        0.20,
        "Existing development; other site constraints still apply",
    ),
    23: (
        "Developed, medium intensity",
        0.15,
        "Existing development; other site constraints still apply",
    ),
    24: (
        "Developed, high intensity",
        0.10,
        "Existing development; other site constraints still apply",
    ),
    31: (
        "Barren land",
        0.30,
        "Unvegetated surface; condition requires local inspection",
    ),
    41: ("Deciduous forest", 0.80, "Caution about vegetation clearance"),
    42: ("Evergreen forest", 0.80, "Caution about vegetation clearance"),
    43: ("Mixed forest", 0.80, "Caution about vegetation clearance"),
    52: ("Shrub/scrub", 0.60, "Potential vegetation disturbance"),
    71: (
        "Grassland/herbaceous",
        0.50,
        "Potential vegetation disturbance; habitat quality unmeasured",
    ),
    81: ("Pasture/hay", 0.40, "Managed vegetated cover; inspect existing use"),
    82: ("Cultivated crops", 0.40, "Managed vegetated cover; inspect existing use"),
    90: (
        "Woody wetlands",
        1.00,
        "High screening caution for disturbance to mapped wetland cover",
    ),
    95: (
        "Emergent herbaceous wetlands",
        1.00,
        "High screening caution for disturbance to mapped wetland cover",
    ),
}
SCENARIOS = {
    "baseline": {
        "ndvi": 0.15,
        "land_cover": 0.25,
        "flood": 0.30,
        "low_elevation": 0.10,
        "slope": 0.20,
    },
    "flood_emphasis": {
        "ndvi": 0.10,
        "land_cover": 0.20,
        "flood": 0.50,
        "low_elevation": 0.05,
        "slope": 0.15,
    },
    "terrain_emphasis": {
        "ndvi": 0.10,
        "land_cover": 0.20,
        "flood": 0.25,
        "low_elevation": 0.15,
        "slope": 0.30,
    },
}
for label, weights in SCENARIOS.items():
    if any(w < 0 for w in weights.values()) or not np.isclose(sum(weights.values()), 1):
        raise ValueError(f"Invalid scenario weights: {label}")


## 3. Boundary and input validation

Check all four input manifests. Land-cover validation includes both parcel IDs and the saved study geometry because its manifest uses a different schema.

In [ ]:
parcels = gpd.read_file(PROJECT_ROOT / "Park_Parcels/Park_Parcles.shp")
if parcels.crs is None:
    raise ValueError("Park parcel CRS is missing.")
parcel_numbers = parcels.Parcel_Num.astype(str).str.strip()
missing = set(PARCEL_IDS) - set(parcel_numbers)
if missing:
    raise ValueError(f"Required parcels missing: {sorted(missing)}")
selected = parcels.loc[parcel_numbers.isin(PARCEL_IDS)].copy()
selected["geometry"] = selected.geometry.map(force_2d).make_valid()
if selected.geometry.is_empty.any() or selected.geometry.isna().any():
    raise ValueError("Selected parcels contain empty geometry.")
preserve_gdf = selected.dissolve().reset_index(drop=True).to_crs(4326)
preserve_m = preserve_gdf.to_crs(METRIC_CRS)
BOUNDARY_HASH = hashlib.sha256(preserve_gdf.geometry.iloc[0].wkb).hexdigest()
metadata = {}
for folder in ["vegetation", "terrain", "flooding"]:
    path = OUTPUT_DIR / folder / "manifest.json"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}; run the corresponding source notebook first."
        )
    record = json.loads(path.read_text())
    if (
        record.get("boundary_hash") != BOUNDARY_HASH
        or sorted(record.get("parcel_ids", [])) != PARCEL_IDS
    ):
        raise ValueError(
            f"{folder} results use a different boundary. Rerun its notebook."
        )
    metadata[folder] = record
lc_dir = OUTPUT_DIR / "land_cover"
lc_manifest = lc_dir / "run_manifest.json"
if not lc_manifest.exists():
    raise FileNotFoundError("Run 01b_Land_Cover_Development.ipynb first.")
metadata["land_cover"] = json.loads(lc_manifest.read_text())
if sorted(metadata["land_cover"].get("parcel_ids", [])) != PARCEL_IDS:
    raise ValueError("Land-cover results use different parcel IDs. Rerun land cover.")
lc_zones = gpd.read_file(lc_dir / "study_zones.geojson").to_crs(METRIC_CRS)
lc_parcel = lc_zones.loc[lc_zones.zone.eq("Selected parcel")]
if (
    lc_parcel.empty
    or lc_parcel.geometry.union_all()
    .symmetric_difference(preserve_m.geometry.iloc[0])
    .area
    > 1
):
    raise ValueError(
        "Saved land-cover study geometry differs from the current parcels."
    )
LC_YEAR = int(metadata["land_cover"]["year"])
LC_PATH = lc_dir / f"land_cover_{LC_YEAR}.tif"
VEGETATION_PATH = (
    OUTPUT_DIR / "vegetation" / metadata["vegetation"]["integration_raster"]
)
TERRAIN_PATH = OUTPUT_DIR / "terrain" / metadata["terrain"]["raster"]
for path in [LC_PATH, VEGETATION_PATH, TERRAIN_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)
print(f"Validated seven parcels; NLCD {LC_YEAR}, summer NDVI 2026.")


## 4. Align inputs to the 30 m terrain grid

Continuous NDVI uses valid-pixel averaging. Categorical land cover uses nearest-neighbor sampling so class codes remain intact. Missing values remain unknown, including areas without FEMA mapping.

In [ ]:
with rasterio.open(TERRAIN_PATH) as src:
    profile = src.profile.copy()
    terrain_data = src.read(masked=True)
    grid_transform, grid_crs = src.transform, src.crs
elevation = terrain_data[0].filled(np.nan)
slope = terrain_data[1].filled(np.nan)
shape = elevation.shape


def align_raster(path, method):
    values = np.full(shape, np.nan, dtype="float32")
    with rasterio.open(path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=values,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=grid_transform,
            dst_crs=grid_crs,
            dst_nodata=np.nan,
            resampling=method,
        )
    return values


vegetation = align_raster(VEGETATION_PATH, Resampling.average)
vegetation[(vegetation < -1) | (vegetation > 1)] = np.nan
land_cover = align_raster(LC_PATH, Resampling.nearest)
land_cover_score = np.full(shape, np.nan, dtype="float32")
for class_code, (_, score, _) in LAND_COVER_RULES.items():
    land_cover_score[land_cover == class_code] = score
flood = gpd.read_file(OUTPUT_DIR / "flooding/fema_zones.geojson").to_crs(grid_crs)
if "SFHA_TF" not in flood or flood.empty:
    raise ValueError("FEMA coverage or SFHA_TF is missing.")


def rasterize_polygons(frame):
    shapes = [(g, 1) for g in frame.geometry if g is not None and not g.is_empty]
    return (
        rasterize(
            shapes, out_shape=shape, transform=grid_transform, fill=0, dtype="uint8"
        )
        if shapes
        else np.zeros(shape, dtype="uint8")
    )


mapped = rasterize_polygons(flood)
sfha = rasterize_polygons(flood.loc[flood.SFHA_TF.astype(str).str.upper().eq("T")])
parcel_mask = np.isfinite(elevation) & np.isfinite(slope)
valid = (
    parcel_mask
    & np.isfinite(vegetation)
    & np.isfinite(land_cover_score)
    & (mapped == 1)
)
if not valid.any():
    raise ValueError("No common valid pixels across all model inputs.")
coverage = pd.DataFrame(
    [
        {
            "input": label,
            "valid_parcel_cells": int((mask & parcel_mask).sum()),
            "pct_terrain_parcel_cells": 100
            * (mask & parcel_mask).sum()
            / parcel_mask.sum(),
        }
        for label, mask in [
            ("NDVI", np.isfinite(vegetation)),
            ("NLCD scored classes", np.isfinite(land_cover_score)),
            ("FEMA mapped coverage", mapped == 1),
            ("Common model area", valid),
        ]
    ]
)
coverage.to_csv(MODEL_DIR / "input_coverage.csv", index=False)
display(coverage)


## 5. Save scoring rules, components, and weights

The score table and scenario weights are exported for review. NDVI, relative elevation, and slope are normalized over the common valid parcel cells. NLCD class scores retain their specified 0–1 values.

In [ ]:
def normalize(values):
    low, high = np.nanmin(values[valid]), np.nanmax(values[valid])
    return np.clip((values - low) / max(high - low, 1e-6), 0, 1)


components = {
    "ndvi": normalize(vegetation),
    "land_cover": land_cover_score,
    "flood": sfha.astype(float),
    "low_elevation": 1 - normalize(elevation),
    "slope": normalize(slope),
}
rules = pd.DataFrame(
    [
        {
            "class_code": key,
            "class_name": value[0],
            "provisional_score": value[1],
            "rationale": value[2],
        }
        for key, value in LAND_COVER_RULES.items()
    ]
)
rules.to_csv(MODEL_DIR / "land_cover_scoring_rules.csv", index=False)
pd.DataFrame.from_dict(SCENARIOS, orient="index").rename_axis("scenario").to_csv(
    MODEL_DIR / "scenario_weights.csv"
)
profile.update(count=1, dtype="float32", nodata=NODATA)


def save_raster(name, values, mask=valid):
    output = np.where(mask, values, NODATA).astype("float32")
    with rasterio.open(MODEL_DIR / name, "w", **profile) as dst:
        dst.write(output, 1)
    return output


save_raster(
    "land_cover_aligned_30m.tif",
    land_cover,
    parcel_mask & np.isfinite(land_cover_score),
)
save_raster("land_cover_component.tif", land_cover_score)
display(rules)
display(pd.DataFrame.from_dict(SCENARIOS, orient="index"))


## 6. Map the land-cover factor and combined scenarios

All constraint maps use a fixed 0–1 color scale. Higher values indicate more screening caution under the stated assumptions. These are parcel-only results.

In [ ]:
def plot_constraint(values, title, filename):
    fig, ax = plt.subplots(figsize=(9, 7))
    show(
        np.ma.masked_equal(values, NODATA),
        transform=grid_transform,
        ax=ax,
        cmap="RdYlBu_r",
        vmin=0,
        vmax=1,
    )
    fig.colorbar(ax.images[0], ax=ax, label="Exploratory screening constraint (0–1)")
    preserve_m.boundary.plot(ax=ax, color="black", linewidth=1)
    ax.set(title=title, xlabel="UTM Easting (m)", ylabel="UTM Northing (m)")
    ax.ticklabel_format(style="plain", useOffset=False)
    fig.text(
        0.5,
        0.01,
        f"Seven parcels • NLCD {LC_YEAR}, Sentinel-2 2026, USGS 3DEP, FEMA • provisional scoring and weights",
        ha="center",
        fontsize=8,
    )
    fig.savefig(MODEL_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


plot_constraint(
    np.where(valid, land_cover_score, NODATA),
    f"NLCD {LC_YEAR} land-cover screening factor",
    "land_cover_component.png",
)
records = []
cell_area_m2 = abs(
    grid_transform.a * grid_transform.e - grid_transform.b * grid_transform.d
)
for label, weights in SCENARIOS.items():
    score = sum(components[key] * weight for key, weight in weights.items())
    output = save_raster(f"constraint_{label}.tif", score)
    if not np.all((output[valid] >= 0) & (output[valid] <= 1)):
        raise ValueError(f"Scores outside 0–1 in {label}.")
    records.append(
        {
            "scenario": label,
            "mean_constraint": float(output[valid].mean()),
            "valid_cells": int(valid.sum()),
            "valid_raster_acres": float(valid.sum() * cell_area_m2 / 4046.8564224),
        }
    )
    plot_constraint(
        output,
        f"Goat Island environmental constraints — {label}",
        f"constraint_{label}.png",
    )
summary = pd.DataFrame(records)
summary.to_csv(MODEL_DIR / "scenario_summary.csv", index=False)
display(summary)


## 7. Record model provenance

The output manifest records every input run and the full score/weight configuration, including the land-cover year.

In [ ]:
manifest = {
    "boundary_hash": BOUNDARY_HASH,
    "parcel_ids": PARCEL_IDS,
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "grid_resolution_m": abs(grid_transform.a),
    "grid_crs": str(grid_crs),
    "grid_transform": list(grid_transform)[:6],
    "inputs": {
        key: {
            "run_utc": value.get("run_utc"),
            "source": value.get("source"),
            "boundary_hash": value.get("boundary_hash"),
        }
        for key, value in metadata.items()
    },
    "land_cover": {
        "year": LC_YEAR,
        "path": str(LC_PATH.relative_to(PROJECT_ROOT)),
        "asset": metadata["land_cover"]["land_cover"]["asset"],
        "product_metadata": metadata["land_cover"]["land_cover"]["metadata"],
        "resampling": "nearest",
        "rules": rules.to_dict(orient="records"),
    },
    "scenarios": SCENARIOS,
    "note": "Provisional disturbance/physical-constraint screening, not validated ecological quality or final access suitability. Unknown inputs excluded.",
}
(MODEL_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print("Saved environmental model:", MODEL_DIR)


## Review before using the results
- Inspect `input_coverage.csv` and the land-cover factor map for alignment or missing-data issues.
- Review and adjust `LAND_COVER_RULES` and `SCENARIOS` for the proposed intervention. These values are assumptions, not calibrated ecological measurements.
- NLCD is 2024 in the current inputs; NDVI is summer 2026. Interpret cross-year disagreements cautiously.
- Weight scenarios provide alternative screening results. Spatial ranking stability and independent validation still need assessment.
- Floodway and zone X subtypes remain visible in the flooding outputs; this model uses SFHA membership only.
- The final access suitability model will also need social/accessibility evidence.